In [34]:
import json
# raw_path = '/mnt/petrelfs/share_data/lizhilin/AnnoData/Temporal-Grounding/qvhl_tvg_cg/qvhl_tvg_cg_tg.json'
raw_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/all_seeing_400k.json'
save_path = '/mnt/petrelfs/share_data/yanziang/qa_data/'
with open(raw_path, 'r') as f:
    raw_data = json.load(f)

In [35]:
raw_data[0]

{'img_path': 'sa_000013/sa_154491.jpg',
 'expression': 'candle holder',
 'bbox': [1679.3447, 864.3755, 1764.1529, 995.0491]}

In [ ]:
new_data = []
for dd in raw_data:
    qa = [{'q':}]

In [29]:
with open(save_path + 'ActivityNet_tvg_cg.json', 'w') as f:
    json.dump(raw_data, f)

In [1]:
import json
import jsonlines
data = []
num = 0
import os
for file in os.listdir('/mnt/petrelfs/share_data/yeshenglong/llava_video/'):
    # print(file)
    if ('youtube' in file ):
        with jsonlines.open('/mnt/petrelfs/share_data/yeshenglong/llava_video/' + file, 'r') as f:
            for dd in f:
                data.append(dd)
len(data)

1117181

In [7]:
data[0]['conversations']

[{'from': 'human',
  'value': '<image>\nWhat items are initially shown inside the transparent glass bottle?\nA. Wooden stick, metal bolt, nut, metal pin, small black ball\nB. Wooden stick, metal bolt, nut, metal pin, small red ball\nC. Wooden stick, metal bolt, screw, metal pin, small white ball\nD. Wooden stick, metal bolt, nut, metal pin, small white ball\nPlease provide your answer by stating the letter followed by the full option.'},
 {'from': 'gpt',
  'value': 'D. Wooden stick, metal bolt, nut, metal pin, small white ball.'},
 {'from': 'human',
  'value': 'What is the first action performed by the hands in the video?\nA. Disassembling the contents of the bottle\nB. Adjusting the position of the wooden stick and metal pin\nC. Placing the items back into the bottle\nD. Rotating the bottle to show different angles\nPlease respond with only the letter of the correct answer.'},
 {'from': 'gpt', 'value': 'A.'},
 {'from': 'human',
  'value': 'Which item is placed into the bottle last dur

In [2]:
f_data = []
for dd in data:
    video = dd['image'].split('public-dataset-p2/')[-1]
    for qa_id in range(len(dd['conversations']) // 2):
        
        q = dd['conversations'][qa_id * 2]
        a = dd['conversations'][qa_id * 2 + 1]
        num = num + 1

        if q['from'] == 'human' and a['from'] == 'gpt' and num % 10 == 0:
            f_data.append({'video': video, 'QA':[{'i':'', 'q':q['value'].split('<image>\n')[-1], 'a':a['value']}]})

In [3]:
len(f_data)

450570

In [4]:
f_data[0]

{'video': 'en-public-llava-video-178k/30_60_s_youtube_v0_1/liwei_youtube_videos/videos/youtube_video_2024/ytb_fYs8TtubjMU.mp4',
 'QA': [{'i': '',
   'q': 'What is the first item placed back into the bottle?\nA. The metal pin\nB. The small white ball\nC. The metal bolt\nD. The wooden stick\nPlease respond with only the letter of the correct answer.',
   'a': 'A.'}]}

In [5]:
with open('/mnt/petrelfs/share_data/yanziang/llava_video/0_30_s_academic_mc_v0_1_qa_processed.json', 'w') as f:
    json.dump(f_data, f)

In [1]:
from petrel_client.client import Client
c = Client('~/petreloss.conf')

In [2]:
for dd in c.list("pnorm2:s3://mevis/masks/"):
    print(dd)
    break

0.json


In [28]:
from petrel_client.version import version

In [9]:
# print(version)
import pycocotools.mask as mask_util


In [14]:
import json
import numpy as np


def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        print(data[0])
        fin_data = []
        for dd in data:
            tt = {}
            tt['video'] = dd['video_path']
            tt['bbox'] = dd['bbox']
            tt['seg_path'] = dd['seg_path']

            template_path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/template/template/tracking4.json'
            with open(template_path, 'r') as f:
                templates = json.load(f)
            rng = np.random.default_rng(None)
            question = rng.choice(templates)
            conversations = [
            {
                'from': 'human', 
                'value': question,
            },
            {
                'from' : 'gpt',
                'value' : f'Answer: <track_begin><tracking> .',
            }
            ]


            tt['conversations'] = conversations
            fin_data.append(tt)
            # print(tt)
            # break
        fin_data = fin_data[:50000]
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/yanziang/trackingllm/videoagent/third-party/segment-anything-2/data/sam5_100k.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/sam5_50k.jsonl')

{'video_path': 'sam-v/sav_train_convert/sav_000/sav_000002.mp4', 'bbox': [639, 255, 982, 623], 'seg_path': 'pnorm2:s3://sam-v/sav_segs/0.json'}


In [32]:
def find_boundaries_torch(mask):
        from skimage.segmentation import find_boundaries
        # mask_np = mask.to(torch.bool).numpy()
        boundaries = find_boundaries(mask, mode='outer')
        boundary_points = np.argwhere(boundaries)
        if boundary_points.size == 0:
            return [-1, -1, -1, -1]
        h0, w0 = boundary_points.min(axis=0)
        h1, w1 = boundary_points.max(axis=0)
        return [float(w0), float(h0), float(w1), float(h1)]

In [4]:
import json
from tqdm import tqdm
import numpy as np
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        print(data[0])
        fin_data = []
        for i, dd in tqdm(enumerate(data)):
            tt = {}
            tt['image'] = dd['img_path'].replace('pnorm2:s3://', '')
            # tt['bbox'] = dd['bbox']
            # tt['seg_path'] = dd['mask']
            
            template_path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/template/template/tracking2.json'
            with open(template_path, 'r') as f:
                templates = json.load(f)
            rng = np.random.default_rng(None)
            question = rng.choice(templates)
            dd['exp'] = dd['expression'][0].lower() + dd['expression'][1:]
            conversations = [
            {
                'from': 'human', 
                'value': question.replace('<obj>', dd['exp']),
            },
            {
                'from' : 'gpt',
                'value' : f'Answer: <track_begin><tracking> .',
            }
            ]
             # mask_util.decode(mask)
            # mask = mask_util.decode(eval(c.get(dd['mask']).decode())['gt_masks'][0])
            # print(mask.shape)
            # print(type(mask))
            # bbox = find_boundaries_torch(mask)
            tt['bbox'] = dd['bbox']
            if tt['bbox'] == [-1,-1,-1,-1]:
                continue
                
            tt['name'] = f'/mnt/petrelfs/share_data/yanziang/internvl/data/allseeing_feature/{i}.pt'
            # print(bbox)
            # print(eval(c.get(dd['mask']).decode())['gt_masks'])


            tt['conversations'] = conversations
            fin_data.append(tt)
            # print(tt)
            # break
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/all_seeing_400k.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/allseeing.jsonl')

{'img_path': 'sa_000013/sa_154491.jpg', 'expression': 'candle holder', 'bbox': [1679.3447, 864.3755, 1764.1529, 995.0491]}


397946it [05:29, 1207.92it/s]


397946


In [31]:
print(len(fin_data))

NameError: name 'fin_data' is not defined

In [4]:
import json
import numpy as np
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        print(data[0])
        fin_data = []
        # try:
        for dd in data:
            try:
                tt = {}
                tt['video'] = dd['video']
                # if dd['expression'] is not []:
                #     dd['expression'] = dd['expression'][0].lower() + dd['expression'][1:]
                conversations = [
                {
                    'from': 'human', 
                    'value': dd['QA'][0]['q']
                },
                {
                    'from' : 'gpt',
                    'value' : dd['QA'][0]['a']
                }
                ]


                tt['conversations'] = conversations
                fin_data.append(tt)
                # if len(fin_data) > 500000:
                #     break
            except:
                print(dd)
            # print(tt)
            # break
        # fin_data = fin_data[:50000]
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

# convert_json_to_jsonl('/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_ref3_train.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/REC_ref3_train.jsonl')
convert_json_to_jsonl('/mnt/petrelfs/share_data/videointern/annotations/anno_instruction/videochat_new/video/reasoning/star/train.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/reason_star.jsonl')

{'video': 'TJZ0P.mp4', 'QA': [{'i': 'Examine the happenings, their sequence, and the interactions in the video. Use this information to decide on the right answer.', 'q': 'Question: Which object was eaten by the person?\nOptions:\n(A) The sandwich.\n(B) The medicine.\n(C) The blanket.\n(D) The box.', 'a': 'Answer: (A) The sandwich.'}], 'start': 7.7, 'end': 15.7}
45731


In [1]:
import json
import numpy as np
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        print(data[0])
        fin_data = []
        for dd in data:
            tt = {}
            tt['video'] = dd['video']
            tt['qid'] = dd['qid']
            tt['vid'] = dd['vid']
            # tt['bbox'] = dd['bbox']
            # tt['height'] = dd['height']
            # tt['width'] = dd['width']
            # tt['seg_path'] = dd['mask']

            conversations = [
            {
                'from': 'human', 
                'value': dd['QA'][0]['q'],
            },
            {
                'from' : 'gpt',
                'value' : 'The given query happens in <time_begin><temp> seconds.',
            }
            ]
            tt['relevant_windows'] = dd['relevant_windows']

            tt['conversations'] = conversations
            fin_data.append(tt)
            # print(tt)
            # break
        fin_data = fin_data[:1000]
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/share_data/lizhilin/AnnoData/Temporal-Grounding/charades_tvg_cg/charades_tvg_cg_train1.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/charades_tvg_cg_train1.jsonl')

{'qid': 0, 'query': 'a person is putting a book on a shelf.', 'duration': 33.67, 'vid': 'AO8RW', 'relevant_windows': [[0.0, 6.9]], 'video': 'AO8RW.mp4', 'QA': [{'q': "Locate and describe the visual content mentioned in the text query 'a person is putting a book on a shelf.' within the video, including timestamps.", 'a': 'The given query happens in 0.0 - 6.9'}]}
1000


In [4]:
import json
import numpy as np
from decord import VideoReader, cpu
from petrel_client.client import Client
import io
import os
from tqdm import tqdm
client = Client()
def get_duration(video_path,  client=None):
    if 's3' in video_path:
        video_bytes = client.get(video_path)
        vr = VideoReader(io.BytesIO(video_bytes), ctx=cpu(0), num_threads=1)
    else:    
        vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
    max_frame = len(vr) - 1
    fps = float(vr.get_avg_fps())
    duration = round(max_frame / fps, 2)
    return duration

def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        # print(data[0])
        fin_data = []
        for dd in tqdm(data):
            try:
                tt = {}
                tt['video'] = dd['video']
                tt['qid'] = dd['qid']
                tt['vid'] = dd['vid']
                # tt['bbox'] = dd['bbox']
                # tt['height'] = dd['height']
                # tt['width'] = dd['width']
                # tt['seg_path'] = dd['mask']

                conversations = [
                {
                    'from': 'human', 
                    'value': dd['QA'][0]['q'],
                },
                {
                    'from' : 'gpt',
                    'value' : 'The given query happens in <time_begin><temp> seconds.' + dd['QA'][0]['a'].split('seconds. ')[1],
                }
                ]
                tt['relevant_windows'] = dd['relevant_windows']

                tt['conversations'] = conversations
                tt['duration'] = get_duration(os.path.join('pssd:s3://anet/ANet_320p/train', dd['video']), client)
                fin_data.append(tt)
            # print(tt)
            # break
            except Exception as e:
                # print(dd)
                ...
        # fin_data = fin_data[:1000]
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/share_data/lizhilin/AnnoData/Temporal-Grounding/ActivityNet_tvg_cg/ActivityNet_tvg_cg-clean.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/ActivityNet_tvg_cg_reasoning.jsonl')

 15%|█▌        | 5111/33467 [04:32<23:26, 20.16it/s] 2025-01-06 16:16:11,853 INF [MainProcess] [MainThread] - MC(id: 139923168327856): No IO operations [P:129829 T:139926680643392 F:io_profile.py:132]
2025-01-06 16:16:11,854 INF [MainProcess] [MainThread] - S3Client(pssd, id: 139923166611296): get [total: 5000, hit: 5000, miss: 0, error: 0, time: 160.475 s, time_avg: 0.032095 s, hit ratio: 100.00%, bytes: 35.2GiB, speed: 224.4MiB/s] [P:129829 T:139926680643392 F:io_profile.py:132]
2025-01-06 16:16:11,854 INF [MainProcess] [MainThread] - S3Client(pvideo, id: 139923080861328): No IO operations [P:129829 T:139926680643392 F:io_profile.py:132]
2025-01-06 16:16:11,855 INF [MainProcess] [MainThread] - S3Client(pnorm2, id: 139923075761728): No IO operations [P:129829 T:139926680643392 F:io_profile.py:132]
2025-01-06 16:16:11,855 INF [MainProcess] [MainThread] - S3Client(pvideo2, id: 139923089544224): No IO operations [P:129829 T:139926680643392 F:io_profile.py:132]
2025-01-06 16:16:11,855 INF

32737


In [2]:
import json
import numpy as np
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        print(data[0])
        fin_data = []
        for dd in data:
            tt = {}
            tt['video'] = dd['video']
            tt['qid'] = dd['qid']
            tt['vid'] = dd['vid']
            tt['duration'] = dd['duration']
            # tt['bbox'] = dd['bbox']
            # tt['height'] = dd['height']
            # tt['width'] = dd['width']
            # tt['seg_path'] = dd['mask']

            conversations = [
            {
                'from': 'human', 
                'value': dd['QA'][0]['q'],
            },
            {
                'from' : 'gpt',
                'value' : 'The given query happens in <time_begin><temp> seconds.',
            }
            ]
            tt['relevant_windows'] = dd['relevant_windows']
            tt['relevant_clip_ids'] = dd['relevant_clip_ids']
            tt['saliency_scores'] = dd['saliency_scores']
            tt['conversations'] = conversations
            fin_data.append(tt)
            # print(tt)
            # break
        # fin_data = fin_data[:50000]
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')

# 示例用法

convert_json_to_jsonl('/mnt/petrelfs/share_data/lizhilin/AnnoData/Temporal-Grounding/qvhl_tvg_cg/qvhl_tvg_cg_tg.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/qvhl_tvg_cg_tg.jsonl')

{'qid': 9769, 'query': 'some military patriots takes us through their safety procedures and measures.', 'duration': 150, 'vid': 'j7rJstUseKg_360.0_510.0', 'relevant_clip_ids': [36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 68, 69, 70, 72], 'saliency_scores': [[4, 3, 2], [4, 1, 3], [4, 3, 4], [4, 1, 2], [4, 2, 2], [4, 2, 2], [4, 2, 2], [4, 2, 2], [4, 1, 2], [4, 1, 3], [4, 3, 4], [4, 3, 3], [4, 3, 3], [4, 3, 4], [4, 3, 2], [4, 1, 2], [4, 3, 2], [4, 1, 2], [4, 1, 2], [4, 1, 2], [4, 1, 2], [3, 3, 3], [4, 3, 4], [4, 1, 3], [4, 1, 2], [4, 1, 2], [4, 1, 2], [4, 1, 2], [4, 1, 2]], 'relevant_windows': [[72, 82], [84, 94], [96, 106], [108, 118], [120, 130], [136, 142], [144, 146]], 'video': 'j7rJstUseKg_360.0_510.0.mp4', 'QA': [{'q': "Localize the visual content described by the given textual query 'some military patriots takes us through their safety procedures and measures.' in the video, and output the start and end timestamps in seconds."

In [31]:
import jsonlines
import json
data = []
with jsonlines.open('/mnt/petrelfs/share_data/yeshenglong/llava_video/0_30_s_academic_mc_v0_1_qa_processed.jsonl', 'r') as f:
    for dd in f:
        dd['video'] = dd['image'].replace('s3://public-dataset-p2/en-public-llava-video-178k/', 'en-public-llava-video-178k/')
        del dd['image']
        data.append(dd)
    with open('/mnt/petrelfs/share_data/yanziang/internvl/data/0_30_s_academic_mc_v0_1_qa_processed.jsonl', 'w', encoding='utf-8') as jsonl_file:
        if isinstance(data, list):
            for item in data:
                jsonl_file.write(json.dumps(item) + '\n')
        else:
            jsonl_file.write(json.dumps(data) + '\n')





In [6]:
import jsonlines
import json
data = []
with jsonlines.open('/mnt/petrelfs/share_data/lixinhao/videochat-next/annotations/video/vqa_tvqa-tvqa_123k.jsonl', 'r') as f:
    for dd in f:
        # dd['video'] = dd['image'].replace('s3://public-dataset-p2/en-public-llava-video-178k/', 'en-public-llava-video-178k/')
        # del dd['image']
        dd['duration'] = round(float(dd['end']) - float(dd['start']),2)
        data.append(dd)
    with open('/mnt/petrelfs/share_data/yanziang/internvl/data/vqa_tvqa-tvqa_123k_with_duration.jsonl', 'w', encoding='utf-8') as jsonl_file:
        if isinstance(data, list):
            for item in data:
                jsonl_file.write(json.dumps(item) + '\n')
        else:
            jsonl_file.write(json.dumps(data) + '\n')





In [5]:
import json
from tqdm import tqdm
import numpy as np
def convert_json_to_jsonl(json_file_path, out_path):
    """
    读取 JSON 文件并将其写入 JSON Lines 文件。

    :param json_file_path: 输入的 JSON 文件路径
    :param out_path: 输出的 JSON Lines 文件路径
    """
    with open(json_file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
        print(data[0])
        fin_data = []
        for i, dd in tqdm(enumerate(data)):
            tt = {}
            tt['video'] = dd['video_path'].replace('pnorm2:s3://', '')
            # tt['bbox'] = dd['bbox']
            # tt['seg_path'] = dd['mask']
            
            template_path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/template/template/tracking2.json'
            with open(template_path, 'r') as f:
                templates = json.load(f)
            rng = np.random.default_rng(None)
            question = rng.choice(templates)
            dd['exp'] = dd['exp'][0].lower() + dd['exp'][1:]
            conversations = [
            {
                'from': 'human', 
                'value': question.replace('<obj>', dd['exp']),
            },
            {
                'from' : 'gpt',
                'value' : f'Answer: <track_begin><tracking> .',
            }
            ]
             # mask_util.decode(mask)
            # mask = mask_util.decode(eval(c.get(dd['mask']).decode())['gt_masks'][0])
            # print(mask.shape)
            # print(type(mask))
            # bbox = find_boundaries_torch(mask)
            # tt['bbox'] = dd['bbox']
            # if tt['bbox'] == [-1,-1,-1,-1]:
            #     continue
                
            # tt['name'] = f'/mnt/petrelfs/share_data/yanziang/internvl/data/allseeing_feature/{i}.pt'
            # print(bbox)
            # print(eval(c.get(dd['mask']).decode())['gt_masks'])
            tt['mask_path'] = dd['mask']

            tt['conversations'] = conversations
            fin_data.append(tt)
            # print(tt)
            # break
        print(len(fin_data))
        with open(out_path, 'w', encoding='utf-8') as jsonl_file:
            if isinstance(fin_data, list):
                for item in fin_data:
                    jsonl_file.write(json.dumps(item) + '\n')
            else:
                jsonl_file.write(json.dumps(fin_data) + '\n')


convert_json_to_jsonl('/mnt/petrelfs/yanziang/trackingllm/videoagent/mevis_val_u3.json', '/mnt/petrelfs/share_data/yanziang/internvl/data/mevis_val.jsonl')

{'video_path': 'pnorm2:s3://mevis/videos/d56a6ec78cfa.mp4', 'exp': 'elephants moving', 'mask': 'pnorm2:s3://mevis/new_mask1/0.json'}


793it [00:00, 1225.10it/s]

793
